In [1]:
print("Test")

Test


In [174]:
from pathlib import Path
import preprocess
import plot
import ims
import numpy as np

input_dir = Path("./data/IMS_Olive_Oil_Geography/Test/test-folder")
output_dir = Path("./output/testing")

dataset = ims.Dataset.read_mea(input_dir)

In [181]:
dataset.files

dataset.interp_riprel()

print(f"Dataset: {dataset}")
print(f"Spectrum-0: {dataset[0]}")

spectrum = dataset[0]

Dataset: Dataset: test-folder, 3 Spectra
Spectrum-0: GC-IMS Spectrum: 151207_144657


In [301]:
# A Spectrum is a 6939 x 3150 numpy.ndarray. (Retention time x drift time)
# Inside are numpy.int16 values.
# Max. Value = 3057
# Min. Value = -180

print("Original ##################################################")
print(f"Spectrum Shape: {spectrum.shape}\n")
print(f"Retention Time")
print(f"Start: {spectrum.ret_time.min()} s  |  End: {spectrum.ret_time.max()} s")
print("-------------------------------------------")
print(f"Drift Time")
print(f"Start: {spectrum.drift_time.min()} ms  |  End: {spectrum.drift_time.max()} ms")
print("\n")

spectrum.cut_rt(start=80, stop=900)
spectrum.cut_dt(start=1.02, stop=2.5)

print("Cutted ##################################################")
print(f"Spectrum Shape: {spectrum.shape}\n")
print(f"Retention Time")
print(f"Start: {spectrum.ret_time.min()} s  |  End: {spectrum.ret_time.max()} s")
print("-------------------------------------------")
print(f"Drift Time")
print(f"Start: {spectrum.drift_time.min()} ms  |  End: {spectrum.drift_time.max()} ms")
print("\n")

Original ##################################################
Spectrum Shape: (6939, 3144)

Retention Time
Start: 0.0 s  |  End: 1019.886 s
-------------------------------------------
Drift Time
Start: 0.0 ms  |  End: 3.0009554136257472 ms


Cutted ##################################################
Spectrum Shape: (5578, 1550)

Retention Time
Start: 79.968 s  |  End: 899.787 s
-------------------------------------------
Drift Time
Start: 1.0197328608820548 ms  |  End: 2.49872743158084 ms


In [176]:
filtered_data = preprocess.preprocess_spectra(input_dir=input_dir,
                                              output_dir=output_dir,
                                              create_plots="None")


Opening dataset: test-folder (3 Spectra)
-------------------------------------------------------
- Spectrum 1/3
- Spectrum 2/3
- Spectrum 3/3


list

In [211]:
# Normalizing the Values

filtered_data[0].values_copy = filtered_data[0].values.copy()
filtered_data[0].values_backup = filtered_data[0].values.copy()

# Data pre normalization
min_val = filtered_data[0].values_copy.min()
max_val = filtered_data[0].values_copy.max()
print(f"Pre-self: {min_val.item(), max_val.item()}")

# Min-Max-Normalisierung auf Wertebereich [0,1] für die Peak-Findung
if max_val - min_val == 0:
    normalized_values = np.zeros_like(filtered_data[0].values_copy)
else:
    normalized_values = (filtered_data[0].values_copy - min_val) / (max_val - min_val)
# Setzen von sample.values auf die normalisierte Matrix für die Peak-Findung.
filtered_data[0].values_copy = normalized_values

min_val = filtered_data[0].values_copy.min()
max_val = filtered_data[0].values_copy.max()
print(f"Post-self: {min_val.item(), max_val.item()}")

filtered_data[0].normalization()
min_val = filtered_data[0].values.min()
max_val = filtered_data[0].values.max()
print(f"Post-Module: {min_val.item(), max_val.item()}")

print(f"Equal normalized: {np.array_equal(filtered_data[0].values, filtered_data[0].values_copy)}")

# Reset the Data
#filtered_data[0].values = filtered_data[0].values_backup

Pre-self: (2.8458510887800914, 679.0236025009776)
Post-self: (0.0, 1.0)
Post-Module: (0.0, 1.0)
Equal normalized: True


In [212]:
# Local maxima tree stuff
import os
import pandas as pd
import multiprocessing
from skimage.morphology import max_tree_local_maxima
from PIL import Image, ImageDraw, ImageFont
import uuid
from matplotlib import pyplot as plt
import seaborn as sns

local_maxima = max_tree_local_maxima(filtered_data[0].values, connectivity=2)

In [284]:
relative_hierarchy_threshold = int(local_maxima.max() * 0.95)

print(f"Shape = {local_maxima.shape}")
print(f"Type of Data = {type(local_maxima[0][0])}")
print(f"Max layer = {local_maxima.max()}")
print(f"Min layer = {local_maxima.min()}")
print(f"Threshold: {relative_hierarchy_threshold}\n")
# Shape = (5578 , 1549)
# Type = unit64
# Max layer = 85434
# Min layer = 0
# Threshold = 81162

Shape = (5578, 1549)
Type of Data = <class 'numpy.uint64'>
Max layer = 85434
Min layer = 0
Threshold: 81162


In [303]:
high_hierarchy_peaks_1 = np.column_stack(
            np.where(
                (local_maxima >= 0.5) &
                (local_maxima <= relative_hierarchy_threshold)
            )
        )

print(high_hierarchy_peaks_1.shape)
# (81162, 2)

min_intensity_threshold = 0.035
high_hierarchy_peaks_2 = np.column_stack(
            np.where(
                (local_maxima >= 0.5) &
                (local_maxima <= relative_hierarchy_threshold) &
                (filtered_data[0].values > min_intensity_threshold)
            )
        )

print(high_hierarchy_peaks_2.shape)
# (6901, 2)
# Interpretation:
# 6901 peaks were found which satisfy the conditions inside of np.where
# Each of these 6901 entries has 2 values
# the Y (retention) and X (drift) coordinate of the found maxima

(81162, 2)
(6901, 2)


In [319]:
peak_intensities_norm = np.array([normalized_values[ret_idx, drift_idx] for ret_idx, drift_idx in high_hierarchy_peaks_2])

In [336]:
print(f"Shape: {peak_intensities_norm.shape}")
print(f"Type: {type(peak_intensities_norm)}")
print(f"Max Intensity: {peak_intensities_norm.max()}")
print(f"Min Intensity: {peak_intensities_norm.min()}")
print(f"Index-0 Intensity: {peak_intensities_norm[0]}\n")
# Shape: (6901,)
# Type: <class 'numpy.ndarray'>
# Max Intensity: 1.0
# Min Intensity: 0.03500036524895548
# Index-0 Value: 0.03630573084691887

# 95.5 percentile berechnen.
# 95.5% der Daten liegen unterhalb dieses Wertes.
intensity_threshold = np.percentile(peak_intensities_norm, 95.5)
print(f"Percentil: {intensity_threshold}")
# 0.05779208307914864

Shape: (6901,)
Type: <class 'numpy.ndarray'>
Max Intensity: 1.0
Min Intensity: 0.03500036524895548
Index-0 Intensity: 0.03630573084691887

Percentil: 0.05779208307914864


In [383]:
# 1. Way to get the coords of the intensity
filtered_peaks = [coord for coord, intensity in zip(high_hierarchy_peaks_2, peak_intensities_norm) if intensity >= intensity_threshold]
highest_hierarchy_peaks = np.array(filtered_peaks)

# 2. Way to get the coords of the intensity
filtered_coords = []
for coord, intensity in zip(high_hierarchy_peaks_2, peak_intensities_norm):
    if(intensity >= intensity_threshold):
        filtered_coords.append(coord)
filtered_coords = np.array(filtered_coords)

print(filtered_coords.shape)
# (311, 2)
filtered_coords

(311, 2)


array([[  67,   49],
       [ 102,    0],
       [ 104,    0],
       [ 165,   62],
       [ 168,   49],
       [ 168,   77],
       [ 169,  164],
       [ 170,    7],
       [ 171,   32],
       [ 171,   83],
       [ 171,  133],
       [ 182,   63],
       [ 242,  139],
       [ 250,   98],
       [ 250,  140],
       [ 250,  228],
       [ 253,    6],
       [ 262,  140],
       [ 288,   32],
       [ 300,   32],
       [ 316,   32],
       [ 337,   32],
       [ 341,  510],
       [ 351,  510],
       [ 359,  503],
       [ 359,  514],
       [ 360,  391],
       [ 366,   99],
       [ 371,  390],
       [ 373,  509],
       [ 374,   33],
       [ 381,   32],
       [ 390,   32],
       [ 397,   32],
       [ 403,   32],
       [ 419,   33],
       [ 432,  100],
       [ 435,   33],
       [ 453,   32],
       [ 457,  100],
       [ 467,   32],
       [ 471,  188],
       [ 474,   33],
       [ 474,  188],
       [ 478,   99],
       [ 489,   32],
       [ 494,  100],
       [ 499,

In [381]:
filtered_data[0].values.shape

(5578, 1549)

In [382]:
valid_peaks = []
min_margin = 5

for coord in highest_hierarchy_peaks:
    ret_index, drift_index = coord

    if (ret_index >= min_margin and ret_index < filtered_data[0].values.shape[0] - min_margin and
            drift_index >= min_margin and drift_index < filtered_data[0].values.shape[1] - min_margin):
        valid_peaks.append(coord)
# die alte Liste wird durch die validierten peaks ersetzt
highest_hierarchy_peaks = np.array(valid_peaks)

highest_hierarchy_peaks

array([[  67,   49],
       [ 165,   62],
       [ 168,   49],
       [ 168,   77],
       [ 169,  164],
       [ 170,    7],
       [ 171,   32],
       [ 171,   83],
       [ 171,  133],
       [ 182,   63],
       [ 242,  139],
       [ 250,   98],
       [ 250,  140],
       [ 250,  228],
       [ 253,    6],
       [ 262,  140],
       [ 288,   32],
       [ 300,   32],
       [ 316,   32],
       [ 337,   32],
       [ 341,  510],
       [ 351,  510],
       [ 359,  503],
       [ 359,  514],
       [ 360,  391],
       [ 366,   99],
       [ 371,  390],
       [ 373,  509],
       [ 374,   33],
       [ 381,   32],
       [ 390,   32],
       [ 397,   32],
       [ 403,   32],
       [ 419,   33],
       [ 432,  100],
       [ 435,   33],
       [ 453,   32],
       [ 457,  100],
       [ 467,   32],
       [ 471,  188],
       [ 474,   33],
       [ 474,  188],
       [ 478,   99],
       [ 489,   32],
       [ 494,  100],
       [ 499,   32],
       [ 513,   32],
       [ 514,

In [390]:
# no replace the normalized data with the non-normalized data.
filtered_data[0].values = filtered_data[0].values_backup

In [394]:
# Erweiterun um nachbarschaftskoordiaten!?
all_coords_to_process = []

# Iteration über jeden Peak aus der gefilterten Liste highest_hierarchy_peak
for ret_index, drift_index in highest_hierarchy_peaks:
    # dx und dy sind Offsets in Retentions- bzw. Driftzeit-Richtung
    # range(-5, 6) bedeutet:Von -5 bis +5 -->Das sind 11 Werte pro Achse --> ergibt 11×11 = 121 Punkte pro Peak
    # new_ret = ret_index + dx; new_drift = drift_index + dy--> Berechnet die neue Koordinate relativ zum ursprünglichen Peak
    #
    for dx in range(-5, 6):
        for dy in range(-5, 6):
            new_ret = ret_index + dx
            new_drift = drift_index + dy
            # Randprüfung--> Nur gültige Koordinaten werden gespeichert
            if (min_margin <= new_ret < filtered_data[0].values.shape[0] - min_margin and
                    min_margin <= new_drift < filtered_data[0].values.shape[1] - min_margin):
                all_coords_to_process.append((new_ret, new_drift))
                
len(all_coords_to_process)

35684

In [399]:
# remove duplicates from all_coords_to_process
all_coords_to_process = list(set(all_coords_to_process))
len(all_coords_to_process)

31637